# Lab 13 — Nemotron-Mini: Chat Template, Multi-Turn Chat & Native Tool Calling

*Part of the NVIDIA Nemotron mini-track inside this Colab lab series — this lab and the ones that follow (09-14 onward) turn to NVIDIA's own Nemotron model family, building on the fine-tuning, quantization, and RAG groundwork from labs 1-12. Unlike those, this one wasn't adapted from an existing course lab — it's new.*

**Runtime setup:** `Runtime > Change runtime type > T4 GPU` (free tier). Works fine on L4/A100 too, just faster.

---

Every open-weight chat model wraps its raw text in *some* templating convention before it ever sees a token — role tags, turn separators, a spot for the system prompt. So far in this track that's mostly meant ChatML-family conventions: `<|im_start|>role ... <|im_end|>` and close relatives like TinyLlama's `<|system|>` / `<|user|>` / `<|assistant|>`. **Nemotron-Mini uses a genuinely different convention**, and this lab exists mostly to make sure that difference doesn't surprise you the first time you hit it.

[**Nemotron-Mini-4B-Instruct**](https://huggingface.co/nvidia/Nemotron-Mini-4B-Instruct) is NVIDIA's small, function-calling-tuned chat model: 4B parameters, distilled and pruned down from the 15B Nemotron-4 base via the Minitron compression recipe (same lineage the sibling Minitron lab, 09-15, covers in more depth — not duplicated here). It's tuned specifically for three things: roleplay, retrieval-augmented QA, and **native function/tool calling** — which makes it a good on-ramp model for the rest of this Nemotron mini-track.

### Key references

- **[Nemotron-4 15B Technical Report (NVIDIA, 2024)](https://arxiv.org/abs/2402.16819)** — the 15B base model this one was distilled from.
- **[Compact Language Models via Pruning and Knowledge Distillation (Muralidharan et al., 2024)](https://arxiv.org/abs/2407.14679)** — the "Minitron" compression technique used to prune Nemotron-4 15B down to the 4B `Minitron-4B-Base` that Nemotron-Mini-4B-Instruct was then fine-tuned from.
- **[Nemotron-Mini-4B-Instruct model card](https://huggingface.co/nvidia/Nemotron-Mini-4B-Instruct)** — architecture specs, license, and the prompt-format documentation this lab follows directly.

### Architecture, in brief

Nemotron-4 decoder-only transformer, embedding size 3072, 32 attention heads, MLP intermediate dimension 9216, Grouped-Query Attention (GQA) + Rotary Position Embeddings (RoPE), 4,096-token context window. In fp16 the weights are ~8 GB — comfortable on a free-tier T4's ~15 GB VRAM, no quantization required.

**Licensing note:** the model card lists this under NVIDIA's open model license (shown as `nvidia-open-model-license` in the repo metadata, and as "NVIDIA Community Model License" in the card body — same non-gated terms either way) and is explicitly cleared for commercial use. **It is not a gated repo** — no `huggingface-cli login`, no access-request click-through, no HF token. Flagging this now because lab 09-17 later in this batch *does* use a gated NVIDIA repo and needs a token — this lab is the easy case.

### What you'll build

1. **Step 1** — load the model, print its raw chat template, and see the `<extra_id_0>` / `<extra_id_1>` format concretely (vs. the ChatML-family tags used earlier in this track).
2. **Step 2** — a multi-turn conversation, watching how turn history accumulates inside the template.
3. **Step 3** — native tool calling end-to-end: define real Python functions, express them as a tool schema, get the model to emit a `<toolcall>`, parse it, execute the function, and feed the result back for a final grounded answer.
4. **Step 4** — a short production-framing note on when a 4B function-calling model like this is the right tool.


In [ ]:
# ============================================================
# Environment & Lab Setup — run this cell first
# ============================================================
# Nemotron-Mini-4B-Instruct is NOT a gated repo, so plain `from_pretrained`
# just works: no huggingface-cli login, no HF_TOKEN, no "request access"
# click-through. (Contrast this with lab 09-17 later in this batch, which
# *does* need a token for a gated NVIDIA repo — worth remembering the
# difference now.)

!pip install -q -U transformers accelerate

import torch

assert torch.cuda.is_available(), "No GPU detected — go to Runtime > Change runtime type > GPU and rerun."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Step 1 — Load Nemotron-Mini and inspect its chat template

Nemotron-Mini's recommended prompt format (from the model card) is:

```
<extra_id_0>System
{system prompt}

<extra_id_1>User
{prompt}
<extra_id_1>Assistant
```

Two things stand out next to the ChatML-family tags this track has used so far:

- The role tag isn't baked into a single token like `<|im_start|>user` — it's a generic sentinel (`<extra_id_0>` for the system turn, `<extra_id_1>` for every turn after it) followed by the role name written as **plain text** on the next line.
- The system turn always opens the prompt, even for a single-turn chat with no system message — the template unconditionally emits `<extra_id_0>System` first.

`<extra_id_0>` / `<extra_id_1>` aren't Nemotron-specific inventions — they're leftover T5-style sentinel tokens repurposed here as turn separators. `tokenizer.apply_chat_template()` builds this string for you from a Jinja template baked into the tokenizer config, so you never have to hand-format it for ordinary chat — but it's worth looking at the raw output once so the format is concrete rather than assumed.

🐛 **Common mistake:** this tokenizer doesn't define a `pad_token` by default. Batch-generate without setting one (`tokenizer.pad_token = tokenizer.eos_token`) and you'll get a `ValueError` about padding. Single-sequence generation (what we do in this lab) doesn't hit it, but it's the first thing to fix if you extend this to batched inference.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "nvidia/Nemotron-Mini-4B-Instruct"

print(f"Loading {MODEL_ID} (~8 GB in fp16, no HF token needed)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token  # not set by default — see the common-mistake note above

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
)
model.eval()
print("Loaded.")
print(f"Context length: {tokenizer.model_max_length} tokens")


In [ ]:
# The raw Jinja template baked into the tokenizer config — this is what
# apply_chat_template() actually renders. Note the unconditional
# "<extra_id_0>System" at the very start, and the {% if tools %} / {% if
# contexts %} branches we'll use in Step 3.
print("=== tokenizer.chat_template (raw Jinja) ===")
print(tokenizer.chat_template)


In [ ]:
# Now render it on an actual message list, so you see the literal string
# the model is conditioned on — this is the "<extra_id_0>/<extra_id_1>"
# format from the model card, concretely.
messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate."},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
]

rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(rendered)


In [ ]:
# Generate. add_generation_prompt=True already appended the
# "<extra_id_1>Assistant\n" cue above, so the model just continues from there.
inputs = tokenizer(rendered, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(reply.strip())


## Step 2 — Multi-turn conversation

The template loops over the full `messages` list, emitting one `<extra_id_1>Role\n...` block per turn (`user`, `assistant`, or `tool`) after the system block. To continue a conversation you just append the model's own reply back into `messages` as an `"assistant"` turn and re-render — the whole history gets re-fed on every call, same as any other chat model without KV-cache reuse across turns.


In [ ]:
def generate_reply(messages, max_new_tokens=120):
    '''Render `messages` with the chat template, generate, decode just the new tokens.'''
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


conversation = [
    {"role": "system", "content": "You are a concise, helpful assistant."},
    {"role": "user", "content": "My favorite city is Kyoto. What's one thing I should see there?"},
]

first_reply = generate_reply(conversation)
print("Assistant:", first_reply)

# Append the model's own turn, then ask a follow-up that only makes sense
# if the earlier turn is actually in context.
conversation.append({"role": "assistant", "content": first_reply})
conversation.append({"role": "user", "content": "Is that within walking distance of the train station?"})

second_reply = generate_reply(conversation)
print("\nAssistant (follow-up):", second_reply)


In [ ]:
# Look at the fully-accumulated prompt string going into the second call —
# every prior turn is a new "<extra_id_1>Role" block, all fed in one shot.
print(tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True))


## Step 3 — Native function/tool calling

This is the centerpiece of the lab. Nemotron-Mini was explicitly fine-tuned to call tools, using this documented format:

```
<extra_id_0>System
{system prompt}

<tool> {tool definition, JSON} </tool>
<context> {optional retrieved context} </context>

<extra_id_1>User
{prompt}
<extra_id_1>Assistant
<toolcall> {"name": ..., "arguments": {...}} </toolcall>
<extra_id_1>Tool
{tool response}
<extra_id_1>Assistant
```

The good news: `tokenizer.apply_chat_template()` already supports this — the Jinja template accepts two extra keyword arguments beyond `messages`:

- **`tools`** — a list of tool definitions. Each one gets serialized straight to JSON and wrapped in its own `<tool> ... </tool>` block.
- **`contexts`** — (plural, not `context`) a list of retrieved-passage strings for RAG-style grounding, each wrapped in `<context> ... </context>`. We won't use this one here, but it's why the tag exists.

So we pass `tools=[...]` directly, the same top-level kwarg used in the standard `transformers` tool-calling API — we don't need to hand-build the string. We'll still check the rendered output and keep a manual fallback, in case a different `transformers`/tokenizer version on your Colab image doesn't forward the kwarg the same way.

The model itself decides whether a tool call is warranted and emits the `<toolcall>{"name": ..., "arguments": {...}}</toolcall>` block as plain generated text — there's no structured `tool_calls` API response like you'd get from a hosted API; **you're responsible for parsing it out of the raw string.** Small models are flaky at this, so we parse defensively and print what we actually saw rather than crashing on a malformed block.


In [ ]:
import ast
import json
import operator
import re


# --- Two toy tools the model can choose to call -----------------------

def get_weather(city: str) -> str:
    '''Stand-in for a real weather API call.'''
    fake_db = {
        "tokyo": "18\u00b0C, light rain",
        "paris": "21\u00b0C, partly cloudy",
        "cairo": "34\u00b0C, clear skies",
        "kyoto": "20\u00b0C, sunny",
    }
    return fake_db.get(city.strip().lower(), f"No data for {city}; assume 20\u00b0C and clear skies.")


def calculate(expression: str) -> str:
    '''Safely evaluate a basic arithmetic expression (no eval()).'''
    ops = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg,
    }

    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ops:
            return ops[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in ops:
            return ops[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expression!r}")

    return str(_eval(ast.parse(expression, mode="eval").body))


AVAILABLE_FUNCTIONS = {"get_weather": get_weather, "calculate": calculate}

# --- Tool schema shown to the model ------------------------------------
# Nemotron-Mini's template just JSON-dumps whatever dict you give it, so
# any reasonable name/description/parameters schema works — this is the
# flat shape (as opposed to OpenAI's nested {"type": "function", ...} wrapper).
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Tokyo' or 'Paris'"},
            },
            "required": ["city"],
        },
    },
    {
        "name": "calculate",
        "description": "Evaluate a basic arithmetic expression and return the numeric result.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "A basic arithmetic expression, e.g. '12 * (4 + 1)'"},
            },
            "required": ["expression"],
        },
    },
]
print(f"Defined {len(AVAILABLE_FUNCTIONS)} callable tools: {list(AVAILABLE_FUNCTIONS)}")


In [ ]:
def render_tool_prompt(messages, tools):
    '''Render messages+tools via apply_chat_template, falling back to a
    hand-built prompt string in Nemotron's documented format if the
    installed tokenizer/transformers version doesn't forward `tools=`.'''
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tools=tools, tokenize=False, add_generation_prompt=True,
        )
        if "<tool>" in prompt:
            return prompt, "apply_chat_template(tools=...)"
    except TypeError as e:
        print(f"apply_chat_template rejected tools=: {e}")

    # --- Manual fallback, following the model card's documented format ---
    system_msg = next((m["content"] for m in messages if m["role"] == "system"), "")
    tool_block = "\n".join(f"<tool> {json.dumps(t)} </tool>" for t in tools)
    parts = [f"<extra_id_0>System\n{system_msg}\n\n{tool_block}\n\n"]
    for m in messages:
        if m["role"] == "user":
            parts.append(f"<extra_id_1>User\n{m['content']}\n")
        elif m["role"] == "assistant":
            parts.append(f"<extra_id_1>Assistant\n{m['content']}\n")
        elif m["role"] == "tool":
            parts.append(f"<extra_id_1>Tool\n{m['content']}\n")
    parts.append("<extra_id_1>Assistant\n")
    return "".join(parts), "hand-built fallback string"


tool_messages = [
    {"role": "system", "content": "You are a helpful assistant with access to tools. Use a tool whenever it would give a more accurate answer than guessing."},
    {"role": "user", "content": "What's the weather like in Tokyo right now?"},
]

tool_prompt, source = render_tool_prompt(tool_messages, tools)
print(f"Prompt built via: {source}\n")
print(tool_prompt)


In [ ]:
def truncate_at_next_turn(text):
    '''Small models sometimes keep talking past their own turn and
    hallucinate the *next* <extra_id_1> block (e.g. a fake Tool reply).
    Cut generation off at the first sign of that so we only keep what the
    model actually said in its own turn.'''
    cut = text.find("<extra_id_1>")
    return text[:cut].strip() if cut != -1 else text.strip()


def parse_toolcall(text):
    '''Pull the JSON payload out of a <toolcall>...</toolcall> block.
    Returns None (and prints what was seen) if nothing parseable is found —
    small models occasionally emit malformed JSON or skip the tags entirely.'''
    match = re.search(r"<toolcall>\s*(\{.*?\})\s*</toolcall>", text, re.DOTALL)
    if not match:
        print("No <toolcall> block found in the model's output. Raw output:")
        print(text)
        return None
    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError as e:
        print(f"Found a <toolcall> block but couldn't parse it as JSON ({e}).")
        print(f"Raw block: {match.group(1)!r}")
        return None


inputs = tokenizer(tool_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)

raw_gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
assistant_turn = truncate_at_next_turn(raw_gen)
print("=== Model's raw turn ===")
print(assistant_turn)

tool_call = parse_toolcall(assistant_turn)
print("\n=== Parsed tool call ===")
print(tool_call)


In [ ]:
# Execute the tool call for real, then feed the result back as a "tool"
# turn and ask the model for its final, grounded answer.
if tool_call is None:
    print("Nothing to execute — the model didn't produce a parseable tool call this run.")
    print("(Small 4B models are flaky here; try re-running the previous cell, or rephrase the prompt.)")
else:
    fn_name = tool_call.get("name")
    fn_args = tool_call.get("arguments", {}) or {}

    if fn_name not in AVAILABLE_FUNCTIONS:
        tool_result = f"Error: unknown function '{fn_name}'"
    else:
        try:
            tool_result = AVAILABLE_FUNCTIONS[fn_name](**fn_args)
        except Exception as e:
            tool_result = f"Error executing {fn_name}: {e}"

    print(f"Executed {fn_name}({fn_args}) -> {tool_result}")

    followup_messages = tool_messages + [
        {"role": "assistant", "content": assistant_turn},
        {"role": "tool", "content": str(tool_result)},
    ]
    followup_prompt, source = render_tool_prompt(followup_messages, tools)

    followup_inputs = tokenizer(followup_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        followup_out = model.generate(**followup_inputs, max_new_tokens=100, do_sample=False)

    final_answer = truncate_at_next_turn(
        tokenizer.decode(followup_out[0][followup_inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    )
    print("\n=== Final grounded answer ===")
    print(final_answer)


In [ ]:
# Same end-to-end flow, different tool — confirms the model is actually
# choosing between tools based on the query, not just pattern-matching one.
calc_messages = [
    {"role": "system", "content": "You are a helpful assistant with access to tools. Use a tool whenever it would give a more accurate answer than guessing."},
    {"role": "user", "content": "What is 17 * (23 + 4)?"},
]

calc_prompt, source = render_tool_prompt(calc_messages, tools)
calc_inputs = tokenizer(calc_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    calc_out = model.generate(**calc_inputs, max_new_tokens=150, do_sample=False)

calc_turn = truncate_at_next_turn(
    tokenizer.decode(calc_out[0][calc_inputs["input_ids"].shape[1]:], skip_special_tokens=False)
)
print("=== Model's raw turn ===")
print(calc_turn)

calc_call = parse_toolcall(calc_turn)
if calc_call is not None and calc_call.get("name") in AVAILABLE_FUNCTIONS:
    result = AVAILABLE_FUNCTIONS[calc_call["name"]](**(calc_call.get("arguments") or {}))
    print(f"\nExecuted {calc_call['name']}({calc_call.get('arguments')}) -> {result}")
else:
    print("\nNo valid tool call parsed for the calculator query this run.")


## Step 4 — Where Nemotron-Mini fits

You now have three data points on this 4B model: it follows a non-ChatML template correctly, it holds multi-turn context, and it can call tools end-to-end. Where does that actually earn its place in a real system, next to the other model sizes you've used in this track?

- **Reach for a small function-calling model like Nemotron-Mini-4B** when the job is a short, well-defined tool loop — one or two tool calls, a fast turnaround, and a deployment target where every GB and every millisecond matters (edge devices, game NPCs — this is literally what NVIDIA built it for with [ACE](https://developer.nvidia.com/ace) — or a latency-sensitive agent hop in a larger pipeline). It's cheap enough to run several in parallel or keep resident alongside other services.
- **Reach for a bigger reasoning-tuned model** when the task needs multi-step planning, longer chains of tool calls, or the tool call itself depends on reasoning through ambiguous intermediate state — that's the job for lab 09-14's **Nemotron-Nano-9B-v2**, covered next in this mini-track.
- **Reach for a general-purpose chat model** (the un-specialized instruction-tuned models used in earlier labs) when tool use isn't the point at all — open-ended conversation, broad world knowledge, or creative writing, where a function-calling fine-tune buys you nothing and may even narrow the model's behavior.

The pattern that transfers regardless of size: know your model's native prompt format, know whether tool calling is templated or hand-built, and always parse defensively — every one of these models can emit a malformed tool call under the wrong sampling settings.


---

## What you just built

A working, end-to-end tour of Nemotron-Mini-4B-Instruct: its non-ChatML `<extra_id_0>`/`<extra_id_1>` chat template inspected directly from the tokenizer config, a multi-turn conversation that correctly carries context across calls, and a full native tool-calling loop — schema definition, model-emitted `<toolcall>`, defensive parsing, real Python execution, and a final answer grounded in the tool's result.

## What to read next

- **[Nemotron-Mini-4B-Instruct model card](https://huggingface.co/nvidia/Nemotron-Mini-4B-Instruct)** — the prompt-format documentation this lab follows, plus AI-safety evaluation notes.
- **[Nemotron-4 15B Technical Report](https://arxiv.org/abs/2402.16819)** — the 15B base model this one was distilled from.
- **[Compact Language Models via Pruning and Knowledge Distillation](https://arxiv.org/abs/2407.14679)** — the Minitron pruning/distillation recipe used to compress it down to 4B (full treatment in lab 09-15).
- **[NVIDIA-NeMo/Nemotron cookbook repo](https://github.com/NVIDIA-NeMo/Nemotron)** — NVIDIA's own `usage-cookbook/` has deployment and tool-calling walkthroughs for other Nemotron sizes; useful for seeing this same pattern at production scale.
- **[NVIDIA ACE: on-device SLM for game character roleplay](https://developer.nvidia.com/blog/deploy-the-first-on-device-small-language-model-for-improved-game-character-roleplay/)** — the real deployment story this model was built for.

## What to try next

- Pass `contexts=[...]` alongside `tools=[...]` to `apply_chat_template` and watch the `<context>` block appear — that's the RAG-plus-tools combination the template was designed for.
- Give the model two tools it could plausibly chain (e.g. `get_weather` then `calculate` a packing suggestion) in one query and see whether it sequences them correctly across turns.
- Swap `get_weather` for a real API call (e.g. `wttr.in`) and see how the model handles a tool result that's messier than the toy dictionary here.
- Move on to **lab 09-14** and run the same tool-calling loop against **Nemotron-Nano-9B-v2** — compare how much more reliably the bigger, reasoning-tuned model parses and sequences the same tool schema.
